In [1]:

from pathlib import Path
import pandas as pd

BASE = Path(r"D:\RouteRanker\data\raw\chennai")
matches = list(BASE.rglob("routes.txt"))

if not matches:
    print("routes.txt not found. Check the extracted folder.")
else:
    GTFS_DIR = matches[0].parent
    data = {}

    for name in ["routes", "stops", "trips", "stop_times"]:
        path = GTFS_DIR / f"{name}.txt"
        if path.exists():
            data[name] = pd.read_csv(path, dtype=str)
            print(name, data[name].shape,
                  "| missing:", data[name].isna().sum().sum())
        else:
            print(name, "file not found")


routes (4611, 5) | missing: 0
stops (5580, 4) | missing: 0
trips (47047, 4) | missing: 0
stop_times (1360443, 5) | missing: 0


In [2]:

# Check duplicate IDs
for name, col in [
    ("routes", "route_id"),
    ("stops", "stop_id"),
    ("trips", "trip_id")
]:
    if name in data and col in data[name].columns:
        print(f"{name} duplicate IDs:",
              data[name][col].duplicated().sum())

# Check relationships
if all(k in data for k in ["routes", "trips"]):
    print("Trips with missing routes:",
          (~data["trips"]["route_id"].isin(
              data["routes"]["route_id"])).sum())

if all(k in data for k in ["trips", "stop_times"]):
    print("Stop times with missing trips:",
          (~data["stop_times"]["trip_id"].isin(
              data["trips"]["trip_id"])).sum())

if all(k in data for k in ["stops", "stop_times"]):
    print("Stop times with missing stops:",
          (~data["stop_times"]["stop_id"].isin(
              data["stops"]["stop_id"])).sum())


routes duplicate IDs: 0
stops duplicate IDs: 0
trips duplicate IDs: 0
Trips with missing routes: 0
Stop times with missing trips: 0
Stop times with missing stops: 0


In [3]:

from pathlib import Path
import pandas as pd

BASE = Path(r"D:\RouteRanker\data\raw\chennai")
matches = list(BASE.rglob("routes.txt"))

if matches:
    GTFS_DIR = matches[0].parent

    for path in sorted(GTFS_DIR.glob("*.txt")):
        df = pd.read_csv(path, dtype=str)
        print(f"\n{path.name}: {len(df)} rows, {len(df.columns)} columns")
        print("Columns:", list(df.columns))
        print("Missing values:", int(df.isna().sum().sum()))
else:
    print("routes.txt not found. Check the Chennai folder.")



agency.txt: 1 rows, 6 columns
Columns: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_phone', 'agency_lang']
Missing values: 0

calendar.txt: 6 rows, 10 columns
Columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
Missing values: 0

feed_info.txt: 1 rows, 8 columns
Columns: ['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_date', 'feed_end_date', 'feed_version', 'feed_contact_email', 'feed_contact_url']
Missing values: 2

routes.txt: 4611 rows, 5 columns
Columns: ['route_id', 'route_short_name', 'route_long_name', 'route_type', 'agency_id']
Missing values: 0

stop_times.txt: 1360443 rows, 5 columns
Columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence']
Missing values: 0

stops.txt: 5580 rows, 4 columns
Columns: ['stop_id', 'stop_name', 'stop_lat', 'stop_lon']
Missing values: 0

trips.txt: 47047 rows, 4 columns
Columns: ['trip_id', 'route_id


## Dataset Relationships

This dataset follows the General Transit Feed Specification (GTFS), which organizes public transit information into related files.

- **routes.txt:** Contains route information. Routes are identified by `route_id`.
- **trips.txt:** Contains scheduled trips. The `route_id` connects a trip to a route, while `service_id` identifies its service schedule.
- **stop_times.txt:** Contains scheduled stop times and stop order for each trip. The `trip_id` connects records to trips, and `stop_id` connects them to stops.
- **stops.txt:** Contains stop identifiers, names, and available location information.
- **calendar.txt:** When present, defines regular service days and operating periods.
- **calendar_dates.txt:** When present, specifies additions or exceptions to regular service dates.

### Relationship Validation

The notebook checks whether trips reference valid routes, stop-time records reference valid trips and stops, and trips have stop-time records. Duplicate identifiers and missing values are also examined.
